## Goal:

Build a simple, interpretable benchmark model to predict `subrogation` (0/1).  
We start with a baseline Logistic Regression and evaluate it using F1.  
This benchmark becomes our reference point before trying more complex models and tuning.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/Users/eugene/Desktop/Emory/Projects/Travelers/data/processed/train_features.csv")
df.head()

,subrogation,email_or_tel_available,safety_rating,high_education_ind,address_change_ind,accident_site,witness_present_ind,liab_prct,channel,policy_report_filed_ind,vehicle_made_year,vehicle_weight,accident_type,in_network_bodyshop,vehicle_mileage,claim_quarter,is_holiday_season,log_claim_est_payout,log_vehicle_price,log_annual_income
0,1,0,75.0,1,1,Parking Area,1,31.0,Broker,1,2021.0,21620.79697,multi_vehicle_clear,no,75421.0,4,1,8.077087,9.697270,11.169970
1,0,1,94.0,1,1,Parking Area,0,34.0,Phone,1,2025.0,10840.58520,multi_vehicle_clear,yes,31988.0,2,0,7.200067,10.437164,11.286326
2,0,1,76.0,1,1,Unknown,0,39.0,Broker,1,2022.0,24318.12282,multi_vehicle_unclear,yes,60876.0,2,0,8.172179,9.615872,10.634123
3,1,1,54.0,1,1,Unknown,0,32.0,Phone,1,2025.0,36958.26656,multi_vehicle_unclear,yes,152772.0,1,0,7.319163,9.740113,10.647803
4,0,1,54.0,1,1,Highway/Intersection,0,28.0,Online,0,2021.0,11779.17422,multi_vehicle_clear,yes,41151.0,1,0,8.533387,10.748212,10.762297


In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["subrogation"])
y = df["subrogation"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# One-hot encode (simple baseline approach)
X_train = pd.get_dummies(X_train, drop_first=True)
X_val   = pd.get_dummies(X_val, drop_first=True)

# Align columns so train/val match
X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)

print("Shapes:", X_train.shape, X_val.shape)

Shapes: (14399, 23) (3600, 23)


In [4]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

In [5]:
y_pred_log = log_reg.predict(X_val)

In [6]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

f1_log = round(f1_score(y_val, y_pred_log), 4)
f1_log

0.5619

In [7]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [8]:
y_pred_xgb = xgb.predict(X_val)
f1_xgb = round(f1_score(y_val, y_pred_xgb), 4)
f1_xgb

0.5046

In [9]:
# import json

# results = {
#     "logistic_f1": f1_log,
#     "xgb_f1": f1_xgb
# }

# with open("/Users/eugene/Desktop/Emory/Projects/Travelers/reports/baseline_results.json", "w") as f:
#     json.dump(results, f)


In [11]:
import json
import os
from datetime import datetime

file_path = "/Users/eugene/Desktop/Emory/Projects/Travelers/reports/baseline_results.json"

new_result = {
    "timestamp": datetime.now().isoformat(),
    "logistic_f1": f1_log,
    "xgb_f1": f1_xgb
}

if os.path.exists(file_path):
    with open(file_path, "r") as f:
        results = json.load(f)

    # 🔑 Convert old dict format to list
    if isinstance(results, dict):
        results = [results]
else:
    results = []

results.append(new_result)

with open(file_path, "w") as f:
    json.dump(results, f, indent=4)

